# Virtual Environments & Python Packaging
Covers: venv, pip, requirements.txt, pyproject.toml, setup.py, __init__.py, package structure, __version__

## 1. Virtual Environment Commands (Shell)

These are shell commands — run them in your terminal, not in Python.

In [ ]:
# Demonstrate venv concepts in Python
import sys
import os

# Check if we're in a virtual environment
in_venv = hasattr(sys, 'real_prefix') or (
    hasattr(sys, 'base_prefix') and sys.base_prefix != sys.prefix
)
print('In virtual environment:', in_venv)
print('Python executable:', sys.executable)
print('sys.prefix:', sys.prefix)

# Show site-packages location
import site
print('\nSite packages:')
for sp in site.getsitepackages():
    print(' ', sp)

## 2. pip — Package Management

In [ ]:
# Use pip programmatically (for inspection)
import subprocess
import sys

# List installed packages
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'list', '--format=columns'],
    capture_output=True, text=True
)
lines = result.stdout.strip().split('\n')
print('Installed packages (first 10):')
for line in lines[:12]:  # header + 10 packages
    print(' ', line)

In [ ]:
# Show package info using importlib.metadata
from importlib.metadata import packages_distributions, version, requires
import importlib.metadata as meta

# Get info about an installed package
package_name = 'pip'
try:
    v = version(package_name)
    print(f'{package_name} version: {v}')
    
    dist = meta.distribution(package_name)
    print(f'Summary: {dist.metadata["Summary"]}')
    print(f'Author: {dist.metadata["Author"]}')
except meta.PackageNotFoundError:
    print(f'{package_name} not found')

## 3. Package Structure — Creating a Package

In [ ]:
import tempfile
import os
from pathlib import Path

# Create a sample package structure in a temp directory
with tempfile.TemporaryDirectory() as tmpdir:
    base = Path(tmpdir)
    
    # Create package structure
    pkg = base / 'src' / 'mypackage'
    pkg.mkdir(parents=True)
    
    # __version__.py
    (pkg / '__version__.py').write_text('__version__ = "1.0.0"\n')
    
    # core.py
    (pkg / 'core.py').write_text('''
def main_function(x, y):
    """Add two numbers."""
    return x + y

class MainClass:
    def __init__(self, name):
        self.name = name
    
    def greet(self):
        return f"Hello from {self.name}"
''')
    
    # utils.py
    (pkg / 'utils.py').write_text('''
def helper(data):
    """A utility function."""
    return [str(item) for item in data]
''')
    
    # __init__.py — the key file!
    (pkg / '__init__.py').write_text('''
"""mypackage — A sample Python package."""

from .__version__ import __version__
from .core import MainClass, main_function
from .utils import helper

__author__ = "Alice Smith"
__all__ = ["MainClass", "main_function", "helper", "__version__"]
''')
    
    # Show structure
    print('Package structure:')
    for path in sorted(base.rglob('*')):
        rel = path.relative_to(base)
        indent = '  ' * (len(rel.parts) - 1)
        print(f'{indent}{path.name}{'/' if path.is_dir() else ""}')
    
    # Test importing from the package
    import sys
    sys.path.insert(0, str(base / 'src'))
    
    import mypackage
    print('\n__version__:', mypackage.__version__)
    print('__all__:', mypackage.__all__)
    
    obj = mypackage.MainClass('Demo')
    print('MainClass.greet():', obj.greet())
    print('main_function(2, 3):', mypackage.main_function(2, 3))
    print('helper([1,2,3]):', mypackage.helper([1, 2, 3]))
    
    # Cleanup sys.path
    sys.path.pop(0)
    del sys.modules['mypackage']

## 4. pyproject.toml — Modern Configuration

In [ ]:
# Show a complete pyproject.toml example
pyproject_toml = '''
[build-system]
requires = ["setuptools>=68.0", "wheel"]
build-backend = "setuptools.backends.legacy:build"

[project]
name = "mypackage"
version = "1.0.0"
description = "A sample Python package"
readme = "README.md"
requires-python = ">=3.9"
dependencies = [
    "requests>=2.28.0",
    "pydantic>=2.0.0",
]

[project.optional-dependencies]
dev = [
    "pytest>=7.0",
    "black>=23.0",
    "mypy>=1.0",
]

[project.scripts]
mypackage-cli = "mypackage.cli:main"

[tool.black]
line-length = 88

[tool.mypy]
python_version = "3.11"
strict = true

[tool.pytest.ini_options]
testpaths = ["tests"]
addopts = "-v --tb=short"
'''

print('pyproject.toml example:')
print(pyproject_toml)

# Parse it with tomllib (Python 3.11+)
try:
    import tomllib
    config = tomllib.loads(pyproject_toml)
    print('Parsed project name:', config['project']['name'])
    print('Parsed version:', config['project']['version'])
    print('Dependencies:', config['project']['dependencies'])
except ImportError:
    print('tomllib available in Python 3.11+')

## 5. __version__ and importlib.metadata

In [ ]:
# Access package version at runtime
from importlib.metadata import version, PackageNotFoundError

def get_version(package_name):
    """Get installed version of a package."""
    try:
        return version(package_name)
    except PackageNotFoundError:
        return 'not installed'

# Check versions of common packages
packages = ['pip', 'setuptools', 'numpy', 'requests', 'flask']
for pkg in packages:
    print(f'{pkg}: {get_version(pkg)}')

# Best practice for __version__ in your package:
# Option 1: __version__.py file
# Option 2: In pyproject.toml, use dynamic versioning
# Option 3: importlib.metadata at runtime

example_init = '''
# In your package __init__.py:
try:
    from importlib.metadata import version
    __version__ = version("mypackage")
except Exception:
    __version__ = "unknown"
'''
print('\nBest practice for __version__:')
print(example_init)

## 6. Interview Q&A Summary

In [ ]:
# Q1: Check if in virtual environment
import sys

def is_in_venv():
    return (
        hasattr(sys, 'real_prefix') or  # virtualenv
        (hasattr(sys, 'base_prefix') and sys.base_prefix != sys.prefix)  # venv
    )

print('In venv:', is_in_venv())

# Q2: requirements.txt best practices
requirements_example = '''
# requirements.txt — direct dependencies with minimum versions
requests>=2.28.0
flask>=3.0.0

# requirements-lock.txt — exact versions for reproducibility (from pip freeze)
requests==2.31.0
certifi==2023.11.17
charset-normalizer==3.3.2
idna==3.6
urllib3==2.1.0
'''
print('requirements.txt example:', requirements_example)

# Q3: setup.py vs pyproject.toml
print('setup.py: legacy, allows arbitrary code (security risk)')
print('pyproject.toml: modern, declarative, PEP 517/518/621 standard')
print('Recommendation: use pyproject.toml for new projects')

# Q4: What does __init__.py do?
print('\n__init__.py:')
print('1. Marks directory as a Python package')
print('2. Exposes public API via imports')
print('3. Sets __version__, __all__, __author__')
print('4. Runs initialization code on import')

# Q5: Editable install
print('\nEditable install: pip install -e .')
print('Creates a link to source — changes reflected immediately')
print('Essential for development workflow')